# Cohort analysis walkthrough

Retention and lifetime value for *Crate & Co*, a fictional UK meal-kit subscription (synthetic data, 2023-2025). All logic lives in `src/cohort_analysis/`; this notebook calls it and explains the results. Run `pip install -e .` first.

In [1]:
import pandas as pd
from cohort_analysis.generate_data import load
from cohort_analysis import cohorts, ltv, sbg, experiments, sql_runner
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 14)
d = load()
c, o = cohorts.prepare(d['customers'], d['orders'])
{k: len(v) for k, v in d.items()}

{'customers': 17818, 'orders': 134773, 'marketing_spend': 180}

## 1. The cohort table
Rows are signup months, columns are months since signup, values are the % of the cohort that received a box. Blank cells have not happened yet.

In [2]:
act = cohorts.activity_retention(c, o)
act.loc['2024-01':'2024-12', :12]

age,0,1,2,3,4,5,6,7,8,9,10,11,12
cohort,,,,,,,,,,,,,
2024-01,100.0,67.8,60.1,55.0,49.7,49.4,46.1,43.8,41.3,39.5,38.7,37.5,35.6
2024-02,100.0,70.9,61.7,54.9,49.3,46.8,43.5,41.4,37.0,35.8,35.1,34.9,34.1
2024-03,100.0,69.3,61.7,55.8,52.6,50.1,48.4,45.1,42.5,42.7,42.5,41.1,36.8
2024-04,100.0,70.7,61.1,56.4,51.4,47.5,46.1,43.2,42.0,39.3,36.1,38.0,33.2
2024-05,100.0,72.4,66.4,60.2,57.6,52.3,49.6,48.0,46.0,44.6,43.2,40.3,37.2
2024-06,100.0,71.2,61.9,56.1,55.6,53.0,48.7,48.5,44.2,44.7,40.9,38.6,36.6
2024-07,100.0,71.5,62.9,55.4,50.2,47.8,47.0,42.8,41.1,39.6,40.8,39.1,38.6
2024-08,100.0,71.0,61.3,56.8,50.5,47.7,47.1,42.3,40.0,38.7,38.2,38.2,35.3
2024-09,100.0,70.6,64.8,59.5,56.1,51.9,51.4,47.5,44.1,42.8,43.1,39.7,39.7


## 2. Average curves
Weighted by cohort size, using only cohorts observed at each age.

In [3]:
sizes = cohorts.cohort_sizes(c)
pd.DataFrame({'activity': cohorts.weighted_curve(act, sizes), 'subscribed': cohorts.weighted_curve(cohorts.survival(c), sizes)}).loc[[1, 3, 6, 12, 24]]

,activity,subscribed
1,72.8,77.4
3,58.7,62.7
6,48.8,51.9
12,37.8,40.3
24,28.0,29.5


## 3. SQL gives the same answer
The same matrix computed in SQLite (`sql/cohort_retention.sql`):

In [4]:
sql = sql_runner.retention_matrix_sql(sql_runner.connect(d))
sql.loc['2024-01':'2024-03', :6]

,0,1,2,3,4,5,6
cohort,,,,,,,
2024-01,100.0,67.8,60.1,55.0,49.7,49.4,46.1
2024-02,100.0,70.9,61.7,54.9,49.3,46.8,43.5
2024-03,100.0,69.3,61.7,55.8,52.6,50.1,48.4


## 4. Segments: offers and channels

In [5]:
cohorts.segment_survival(c, 'first_box_offer')

,first_box_offer,customers,month_1_pct,month_3_pct,month_6_pct,month_12_pct
2,No offer,6697,85.9,71.0,60.4,48.8
0,50% off first box,6818,82.3,66.4,53.7,41.3
1,First box free,4303,56.6,44.5,35.5,25.8


In [6]:
cohorts.segment_survival(c, 'channel')

,channel,customers,month_1_pct,month_3_pct,month_6_pct,month_12_pct
3,Referral,2062,83.6,73.0,65.5,56.1
1,Organic,3370,80.9,68.3,57.1,46.6
4,Search,4069,77.6,63.5,53.3,41.4
0,Influencer,3914,74.2,57.5,44.3,30.1
2,Paid social,4403,74.3,57.3,46.2,34.6


## 5. Lifetime value vs acquisition cost
Contribution = revenue minus cost of goods (a free box still costs money).

In [7]:
ltv.ltv_cac_table(c, o, d['marketing_spend'])

,channel,cac,ltv_12m,ltv_12m_to_cac,ltv_24m,payback_month
1,Organic,6.06,100.83,16.64,172.30,1
3,Referral,18.12,112.83,6.23,199.50,2
4,Search,38.33,91.46,2.39,155.60,5
2,Paid social,46.51,80.97,1.74,134.09,6
0,Influencer,58.11,74.65,1.28,103.03,9


In [8]:
ltv.offer_economics(c, o)

,first_box_offer,first_box_contribution,contribution_3m,contribution_12m
2,No offer,14.34,48.53,122.82
0,50% off first box,-6.14,26.64,91.46
1,First box free,-26.61,-4.41,37.07


## 6. Calendar effect: the April 2025 price rise

In [9]:
cohorts.calendar_churn(c).query('month >= "2025-01" and month <= "2025-07"')

,month,subscribers_start,cancellations,churn_rate_pct,tenured_churn_rate_pct
23,2025-01,4814,283,5.88,4.80
24,2025-02,5366,407,7.58,4.33
25,2025-03,5663,356,6.29,4.56
26,2025-04,5949,682,11.46,9.85
27,2025-05,5839,335,5.74,4.37
28,2025-06,6079,321,5.28,3.92
29,2025-07,6304,326,5.17,3.92


## 7. Forecasting with the shifted-beta-geometric model
Fit on 6 months of the Jan-Jun 2023 cohorts, then compare the forecast with what actually happened.

In [10]:
bt = sbg.backtest(c, [f'2023-0{i}' for i in range(1, 7)])
bt['params'], bt['sbg_mape_pct'], bt['constant_churn_mape_pct']

({'a': 0.3,
  'b': 0.9099,
  'mean_churn_month1': 0.248,
  'neg_log_lik': 3067.07,
  'converged': True},
 7.2,
 59.9)

In [11]:
bt['curve'].iloc[[1, 3, 6, 12, 18, 24, 30]]

,month,actual,sbg_forecast,constant_churn_forecast
1,1,75.1,75.2,88.8
3,3,59.1,58.9,70.0
6,6,49.0,49.1,49.0
12,12,39.4,40.4,24.0
18,18,34.4,35.9,11.8
24,24,29.6,33.0,5.8
30,30,26.7,30.9,2.8


## 8. Did the new onboarding (Sep 2024) work?

In [12]:
experiments.onboarding_effect(c)

{'window': '2023-09 to 2025-08',
 'outcome': 'still subscribed at month 3',
 'n_before': 5177,
 'n_after': 7132,
 'naive': {'before_pct': 60.4,
  'after_pct': 65.4,
  'difference_pts': 5.0,
  'ci_low_pts': np.float64(3.2),
  'ci_high_pts': np.float64(6.7),
  'z': 5.66,
  'p_value': 0.0},
 'mix_shift': {'influencer_pct_before': 19.6,
  'influencer_pct_after': 24.2,
  'free_box_pct_before': 23.9,
  'free_box_pct_after': 23.8},
 'adjusted': {'odds_ratio': 1.275,
  'ci_low': 1.181,
  'ci_high': 1.376,
  'p_value': 0.0,
  'adjusted_difference_pts': 5.3}}

Charts, CSVs and the interactive dashboard are produced by `python -m cohort_analysis analyse`.